# Validate Privacy Guarantees with Auditing

Privacy accounting gives *theoretical* upper bounds on privacy loss.
Privacy auditing gives *empirical* lower bounds by running membership
inference attacks against the trained model. If the audited epsilon
exceeds the theoretical bound, there is likely a bug in the
implementation.

This notebook walks through the auditing workflow: constructing an
`AuditResult` from attack scores, estimating epsilon, computing
attack metrics, and building confidence intervals for AUC.

**Prerequisites:** Familiarity with (epsilon, delta)-DP.

**Components exercised:** `AuditResult`,
`epsilon_at`, `auc`, `beta_at`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from opaque.auditing import AuditResult
from opaque.random import key

np.random.seed(42)

## Simulated attack scores

In practice these come from a membership inference attack. Here we
simulate three scenarios with increasing separation between in-members
and out-members.

In [ ]:
n = 500

scenarios = {
    "good_dp": (
        np.random.normal(0.55, 0.30, n),
        np.random.normal(0.45, 0.30, n),
    ),
    "weak_dp": (
        np.random.normal(0.70, 0.25, n),
        np.random.normal(0.30, 0.25, n),
    ),
    "no_dp": (
        np.random.normal(0.90, 0.10, n),
        np.random.normal(0.10, 0.10, n),
    ),
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, (in_s, out_s)) in zip(axes, scenarios.items()):
    ax.hist(in_s, bins=30, alpha=0.6, label="In", density=True)
    ax.hist(out_s, bins=30, alpha=0.6, label="Out", density=True)
    ax.set_title(name)
    ax.set_xlabel("Score")
    ax.legend()
plt.tight_layout()
plt.show()

## Epsilon estimation

`AuditResult.epsilon_at` converts the empirical TPR/FPR tradeoff into
a lower bound on epsilon using the one-run likelihood-ratio test
(Steinke et al., 2023).

In [ ]:
delta = 1e-5

for name, (in_s, out_s) in scenarios.items():
    r = AuditResult(in_s, out_s)
    eps = r.epsilon_at(delta=delta)
    print(f"{name:8s}  epsilon: {eps:.2f}")

## Attack metrics

In [ ]:
for name, (in_s, out_s) in scenarios.items():
    r = AuditResult(in_s, out_s)
    print(
        f"{name:8s}  "
        f"AUC={r.auc():.3f}  "
        f"β@α=0.01={r.beta_at(alpha=0.01):.3f}  "
        f"Accuracy={r.max_accuracy():.1%}"
    )

## Summary report

In [ ]:
r = AuditResult(*scenarios["weak_dp"])
print(r.summary(significance=0.05, delta=1e-5))

## AUC confidence intervals

`AuditResult.auc` supports confidence intervals directly.
Pass `confidence=` and `key=` to get a `(lower, upper)` tuple instead
of a point estimate.

In [ ]:
r = AuditResult(*scenarios["weak_dp"])

# Point estimate
auc_point = r.auc()

# 95% confidence interval
auc_ci = r.auc(confidence=0.95, num_samples=1000, key=key(42))

print(f"AUC:     {auc_point:.3f}  95% CI [{auc_ci[0]:.3f}, {auc_ci[1]:.3f}]")
print(f"Epsilon: {r.epsilon_at(delta=1e-5):.2f}")

## Complete audit workflow

In [ ]:
def audit(in_scores, out_scores, theoretical_epsilon, delta=1e-5):
    """Run a privacy audit and compare to the theoretical bound."""
    result = AuditResult(in_scores, out_scores)

    eps = result.epsilon_at(delta=delta, significance=0.05)
    auc_ci = result.auc(confidence=0.95, key=key(42))

    # Use summary() with theoretical_epsilon for side-by-side comparison
    print(result.summary(delta=delta, theoretical_epsilon=theoretical_epsilon))
    print()
    print(f"AUC 95% CI: [{auc_ci[0]:.3f}, {auc_ci[1]:.3f}]")

    if eps > theoretical_epsilon:
        print("WARNING: audited epsilon exceeds theoretical bound.")
    else:
        print("OK: audited epsilon is within the theoretical bound.")


audit(*scenarios["weak_dp"], theoretical_epsilon=8.0)
print()
audit(*scenarios["good_dp"], theoretical_epsilon=3.0)

## Epsilon estimation vs sample size

Larger audit sets produce tighter estimates.

In [ ]:
in_full = np.random.normal(0.7, 0.25, 2000)
out_full = np.random.normal(0.3, 0.25, 2000)

sizes = [50, 100, 200, 500, 1000]
eps_vals = []

for sz in sizes:
    r = AuditResult(in_full[:sz], out_full[:sz])
    eps_vals.append(r.epsilon_at(significance=0.05))

plt.figure(figsize=(8, 5))
plt.plot(sizes, eps_vals, "o-", label="One-run")
plt.xlabel("Canaries per group")
plt.ylabel("Estimated epsilon")
plt.title("Epsilon estimation vs sample size")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

1. Build an `AuditResult` from attack in/out scores.
2. Estimate epsilon with `epsilon_at` (or `epsilon_one_run` for
   explicit control).
3. Inspect `auc`, `beta_at`, and `max_accuracy` for attack
   strength.
4. Quantify AUC uncertainty with `auc(confidence=0.95, key=...)`.
5. Compare audited epsilon to the theoretical bound using
   `summary(theoretical_epsilon=...)`.

For training integration, see the
[Privacy Auditing guide](../user-guide/auditing.md) which shows
how to use `auditing.setup()`, `auditing.evaluate()`, and the
`batch_argnums` / `batch_unpack` pattern for HuggingFace models.